# CysRank 可复现训练与位点筛选

本 Notebook 演示 **数据核对 → 训练与参数优化 → 诊断 → 已有 Cys 位点评分 → Top 3 导出**。
使用项目提供的小规模实验来源数据，调用正式 Python 入口，不复制另一套训练算法。

模型为冻结的 ProteinMPNN 编码器加线性分类头。当前模型只评估已有 Cys，**不生成插入新 Cys 的序列**。
候选为训练回代演示，评分不是校准成功率或反应效率；母蛋白留出仅 0/2 命中，无独立阴性测试。

## 开始前

1. 在仓库根目录执行 `python -m pip install -r requirements-notebook.txt`。
2. 使用同一环境运行 `python -m jupyterlab`，打开本文件并选择该环境的 Python 内核。
3. 选择 **Restart Kernel and Run All Cells**。默认训练 50 轮，可在参数单元格改为 1 轮冒烟检查。

从仓库根目录或 notebooks 目录启动均可自动定位。若内核工作目录在仓库之外，先设置环境变量
`CYSRANK_ROOT` 指向移植后的仓库目录。后续数据、模型和输出路径全部基于识别出的仓库根目录。

In [1]:
from pathlib import Path
import os
import sys

hint = os.environ.get("CYSRANK_ROOT")
start = Path(hint).expanduser().resolve() if hint else Path.cwd().resolve()
roots = [start] if hint else [start, *start.parents]
ROOT = next((p for p in roots if all((p / name).is_file() for name in
    ("configs/train.json", "src/paths.py", "predict.py", "run.py"))), None)
if ROOT is None:
    raise RuntimeError("请从仓库内启动内核，或先设置 CYSRANK_ROOT 为仓库目录。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.paths import root_path, ROOT as CODE_ROOT
assert CODE_ROOT == ROOT, "当前内核加载了另一个项目，请重启内核。"
print("已定位仓库根目录；所有相对路径以此为基准。")

已定位仓库根目录；所有相对路径以此为基准。


In [2]:
import csv
import json
import html
import platform
import subprocess
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np
import torch
from IPython.display import display, HTML
from src import core, pipeline
from predict import export

def read_json(path):
    return Path(path).read_text(encoding="utf-8-sig")

def table(rows):
    if not rows:
        display(HTML("<p>没有符合条件的记录。</p>"))
        return
    columns = list(rows[0])
    def cell(value):
        return html.escape(str(value))
    header = "".join("<th>" + cell(k) + "</th>" for k in columns)
    body = "".join("<tr>" + "".join("<td>" + cell(r.get(k, "")) + "</td>" for k in columns) + "</tr>" for r in rows)
    display(HTML("<table><thead><tr>" + header + "</tr></thead><tbody>" + body + "</tbody></table>"))

table([{"Python": platform.python_version(), "PyTorch": str(torch.__version__),
        "NumPy": np.__version__, "CUDA可用": torch.cuda.is_available()}])

Python,PyTorch,NumPy,CUDA可用
3.13.5,2.11.0+cu128,2.5.2,True


## 1. 参数和输出位置

输入为项目随附的无标签特征文件；`feature_mask` 只选择特征已准备的 Cys。
训练读取 configs/train.json 指定的正负记录和母蛋白分组信息，与推理输入分别管理。
每次运行此单元格都会创建新的运行编号，重复 Run All 不覆盖正式模型和结果。
演示赛道必须在正式提交前替换为真实赛道。

In [3]:
EPOCHS = 50
TOP_K = 3
TRACK = "未指定（演示）"
INPUT = "data/examples/input.jsonl"
CHECKPOINT = "models/site_predictor.pt"
assert isinstance(EPOCHS, int) and EPOCHS > 0
assert isinstance(TOP_K, int) and TOP_K > 0
config = json.loads(read_json(root_path("configs/train.json")))
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid4().hex[:8]
RUN = root_path("logs/notebook") / run_id
RUN.mkdir(parents=True, exist_ok=False)
print("本次输出：", RUN.relative_to(ROOT).as_posix())
print("轮数：", EPOCHS, "；随机种子：", config["seed"])

本次输出： logs/notebook/20260916T103459_f64be85d
轮数： 50 ；随机种子： 20260916


## 2. 数据核对

原正负文件的 ca_label 表示已验证位点掩码，正负结局由文件来源确定。
不把未标注 Cys 当作负例；外部验证文件、测试 PDB 和模拟数据不进入此次损失。
以下统计来自实际预处理，输入不含监督标签。

In [4]:
training = pipeline.prepare(root_path(config["source"]), root_path(config["parents"]))
examples = core.load_jsonl(root_path(INPUT))
for row in examples:
    core.validate(row)
assert len({r["record_id"] for r in examples}) == len(examples)
assert all("ca_label" not in r for r in examples)
stats = {
    "训练记录": len(training),
    "监督位点": sum(sum(r["label_mask"]) for r in training),
    "正位点": sum(sum(r["ca_label"]) for r in training),
    "负位点": sum(sum(r["confirmed_negative_mask"]) for r in training),
    "推断母蛋白组": len({r["parent_group"] for r in training}),
    "推理示例记录": len(examples),
}
table([stats])
table([{ "记录": r["record_id"], "序列": r["seq"], "长度": len(r["seq"]),
         "可评分Cys数": sum(aa == "C" and bool(m) for aa, m in zip(r["seq"], r["feature_mask"]))}
       for r in examples[:5]])

训练记录,监督位点,正位点,负位点,推断母蛋白组,推理示例记录
16,19,5,14,3,16


记录,序列,长度,可评分Cys数
positive_001,ECG,3,1
positive_002,RQRLRCASIQK,11,1
positive_003,VHKECCHGDLL,11,1
positive_004,KLKECCDKPLL,11,1
positive_005,DVQAVCSQKNV,11,1


## 3. 训练、优化与筛选

调用正式 run.py，使用内核的同一解释器。冻结编码器并用 AdamW 优化分类头；
保持固定轮次和阈值，不根据留出结果选模型。本次训练写入独立目录，**不会替换 models/site_predictor.pt**。
完整 stdout/stderr 保存到运行目录，失败时抛出异常停止流程。

In [5]:
workflow_dir = RUN / "workflow"
command = [sys.executable, str(root_path("run.py")), "--output",
           workflow_dir.relative_to(ROOT).as_posix(), "--epochs", str(EPOCHS), "--track", TRACK]
completed = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                           encoding="utf-8", errors="replace")
(RUN / "workflow.log").write_text(completed.stdout + completed.stderr, encoding="utf-8")
if completed.returncode:
    raise RuntimeError("训练流程失败，请查看 " + (RUN / "workflow.log").relative_to(ROOT).as_posix()
                       + "\n" + completed.stderr[-3000:])
summary = json.loads(read_json(workflow_dir / "training/summary.json"))
timing = json.loads(read_json(workflow_dir / "timing.json"))
table([{"训练轮数": EPOCHS, "耗时秒": round(timing["elapsed_seconds"], 2),
        "训练位点": summary["training_sites"], "模型重新加载通过": summary["reload_verified"]}])

训练轮数,耗时秒,训练位点,模型重新加载通过
50,3.38,19,True


## 4. 训练结果与分组诊断

训练回代衡量拟合情况，不能代表泛化。母蛋白分组由精确序列包含关系推断，尚不等同于完整同源去重。
含全部阴性的母蛋白组留出后，训练只剩正类，因此该折跳过；其余单类留出无法估计总体准确率或误报率。
下方同时保留每折信息和损失日志片段，完整 JSON 在本次训练目录中。

In [6]:
table([{"范围": "训练回代（原始）", **summary["training"]["raw"]},
       {"范围": "训练回代（加过滤）", **summary["training"]["filtered"]}])
diagnostic = summary["diagnostics"]["full"]
table([{"留出母蛋白": f["parent"], "状态": f["status"],
        "训练位点": f["train_n"], "留出位点": f["held_n"],
        "召回率": f.get("raw", {}).get("recall", "不适用")}
       for f in diagnostic["folds"]])
print("合并留出指标：")
display(diagnostic["pooled"])
loss_log = json.loads(read_json(workflow_dir / "training/training_log.json"))
print("损失日志首尾记录：")
display(loss_log[:2] + loss_log[-2:] if len(loss_log) > 4 else loss_log)

范围,n,positive_sites,negative_sites,tp,fp,tn,fn,accuracy,precision,recall,specificity,f1,average_precision,threshold,note
训练回代（原始）,19,5,14,5,0,14,0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,"Average precision is threshold-grouped AP, not trapezoidal PR-AUC."
训练回代（加过滤）,19,5,14,5,0,14,0,1.0,1.0,1.0,1.0,1.0,None,0.5,"Average precision is threshold-grouped AP, not trapezoidal PR-AUC."


留出母蛋白,状态,训练位点,留出位点,召回率
old3_positive_03,evaluated,18,1,0.0
old3_positive_06,skipped_single_class_training,2,17,不适用
old3_positive_07,evaluated,18,1,0.0


合并留出指标：


{'n': 2,
 'positive_sites': 2,
 'negative_sites': 0,
 'tp': 0,
 'fp': 0,
 'tn': 0,
 'fn': 2,
 'accuracy': None,
 'precision': None,
 'recall': 0.0,
 'specificity': None,
 'f1': None,
 'average_precision': None,
 'threshold': 0.5,
 'note': 'Single-class sets cannot estimate both sensitivity and specificity; AP omitted.'}

损失日志首尾记录：


[{'epoch': 1,
  'weighted_bce': 0.5794527530670166,
  'n': 19,
  'positive_sites': 5,
  'negative_sites': 14,
  'tp': 0,
  'fp': 0,
  'tn': 14,
  'fn': 5,
  'accuracy': 0.7368421052631579,
  'precision': None,
  'recall': 0.0,
  'specificity': 1.0,
  'f1': 0.0,
  'average_precision': 0.8711111111111111,
  'threshold': 0.5,
  'note': 'Average precision is threshold-grouped AP, not trapezoidal PR-AUC.'},
 {'epoch': 2,
  'weighted_bce': 0.5039470791816711,
  'n': 19,
  'positive_sites': 5,
  'negative_sites': 14,
  'tp': 0,
  'fp': 0,
  'tn': 14,
  'fn': 5,
  'accuracy': 0.7368421052631579,
  'precision': None,
  'recall': 0.0,
  'specificity': 1.0,
  'f1': 0.0,
  'average_precision': 0.8711111111111111,
  'threshold': 0.5,
  'note': 'Average precision is threshold-grouped AP, not trapezoidal PR-AUC.'},
 {'epoch': 49,
  'weighted_bce': 0.06151604652404785,
  'n': 19,
  'positive_sites': 5,
  'negative_sites': 14,
  'tp': 5,
  'fp': 0,
  'tn': 14,
  'fn': 0,
  'accuracy': 1.0,
  'precision

## 5. 随附最终模型的 Top 3

正式提交权重与刚训练的模型分开使用。本单元格加载 models/site_predictor.pt，调用项目统一导出函数。
仅分数 ≥ 0.5 且通过二硫键、前位 Pro 过滤的位点进入全局排序；同分按记录编号及位点顺序排列。
分数不等于反应效率。没有对应全原子结构时，结构字段留空，不把测试 PDB 错配给候选。

In [7]:
# 独立编号也允许只重复执行本单元格，而不覆盖前次导出。
candidate_path = RUN / ("delivered_model_" + uuid4().hex[:8] + ".csv")
export(root_path(INPUT), root_path(CHECKPOINT), candidate_path, TRACK, TOP_K)
with candidate_path.open(encoding="utf-8", newline="") as stream:
    candidates = list(csv.DictReader(stream))
columns = ["candidate_id", "sequence", "position_1based", "prediction_score", "rank",
           "is_disulfide", "sasa_A2", "prev_is_pro", "nearby_atoms_5A"]
table([{k: r[k] for k in columns} for r in candidates])
print("候选 CSV：", candidate_path.relative_to(ROOT).as_posix())
print("全位点审计：", candidate_path.with_suffix(".sites.jsonl").relative_to(ROOT).as_posix())

D:\CysRank-GCLSC\src\protein_mpnn_utils.py:799: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Cross.cpp:67.)
  n_2 = F.normalize(torch.cross(u_2, u_1), dim=-1)


Scored 19 existing Cys sites; scores are not calibrated efficiencies.
Exported 3 candidates to delivered_model_83cef1d8.csv


candidate_id,sequence,position_1based,prediction_score,rank,is_disulfide,sasa_A2,prev_is_pro,nearby_atoms_5A
positive_005_C6,DVQAVCSQKNV,6,0.98269147,1,0,0.04,0,26
positive_001_C2,ECG,2,0.96828640,2,0,141.8,0,8
positive_004_C6,KLKECCDKPLL,6,0.95841920,3,0,28.12,0,28


候选 CSV： logs/notebook/20260916T103459_f64be85d/delivered_model_83cef1d8.csv
全位点审计： logs/notebook/20260916T103459_f64be85d/delivered_model_83cef1d8.sites.jsonl


## 6. 复现核验

按记录编号和位点匹配历史预测，不依赖文件行顺序。对默认数据和随附权重，
分数差应在浮点误差范围内；若换用自有输入或模型，则不与原始样例强行比较。
新训练结果也与原始训练摘要对照。改为少量轮次后，不要求拟合结果与 50 轮模型一致。

In [8]:
current = core.load_jsonl(candidate_path.with_suffix(".sites.jsonl"))
report = {"epochs": EPOCHS, "seed": config["seed"], "exported_candidates": len(candidates),
          "scored_sites": len(current), "run_directory": RUN.relative_to(ROOT).as_posix()}
if INPUT == "data/examples/input.jsonl" and CHECKPOINT == "models/site_predictor.pt":
    historical = core.load_jsonl(root_path("logs/original/final_model/reload_predictions.jsonl"))
    key = lambda r: (r["record_id"], r["position_1based"])
    expected = {key(r): r for r in historical}
    assert {key(r) for r in current} == set(expected)
    delta = max((abs(r["raw_score"] - expected[key(r)]["raw_score"]) for r in current), default=0.0)
    assert delta <= 1e-6, f"分数差异超出容差：{delta}"
    assert all(r["predicted_positive"] == expected[key(r)]["predicted_positive"] for r in current)
    report["max_delivered_model_score_difference"] = delta
original_summary = json.loads(read_json(root_path("logs/original/summary.json")))
report["training_metrics_match_original"] = summary["training"] == original_summary["training"]
report["grouped_diagnostics_match_original"] = summary["diagnostics"] == original_summary["diagnostics"]
core.write_json(RUN / "notebook_verification.json", report)
display(report)
print("新训练模型的候选示例：")
with (workflow_dir / "results.csv").open(encoding="utf-8", newline="") as stream:
    retrained_candidates = list(csv.DictReader(stream))
table([{k: r[k] for k in columns} for r in retrained_candidates])

{'epochs': 50,
 'seed': 20260916,
 'exported_candidates': 3,
 'scored_sites': 19,
 'run_directory': 'logs/notebook/20260916T103459_f64be85d',
 'max_delivered_model_score_difference': 0.0,
 'training_metrics_match_original': True,
 'grouped_diagnostics_match_original': True}

新训练模型的候选示例：


candidate_id,sequence,position_1based,prediction_score,rank,is_disulfide,sasa_A2,prev_is_pro,nearby_atoms_5A
positive_005_C6,DVQAVCSQKNV,6,0.98269147,1,0,0.04,0,26
positive_001_C2,ECG,2,0.96828640,2,0,141.8,0,8
positive_004_C6,KLKECCDKPLL,6,0.95841920,3,0,28.12,0,28


## 结果文件与解释

本次目录 logs/notebook/<运行编号>/ 中包含：

- workflow/training/：新训练权重、配置、日志、分组消融及处理后数据。
- workflow/results.csv：新训练模型的候选示例及伴随审计文件。
- delivered_model_*.csv：原随附最终模型的候选示例及伴随审计文件。
- workflow.log、notebook_verification.json：运行日志和复现核验。

所有结果均基于训练来源示例，不是独立实验验证。默认的 50 轮训练回代可拟合 19 个位点，
但母蛋白留出仅 0/2 命中；没有独立阴性测试，也未提供可靠置信区间。
来源许可、全原子结构特征的获取过程及预训练重叠审计仍需补齐。
参见仓库中的 data/README.md、models/MODEL_CARD.md 和 docs/SUBMISSION_CHECKLIST.md。